In [1]:
# build simple agent to understand agentic process
# syntax
""" While goal not done:
    think()
    act()
    observe()
    update_memory()
    ..."""


' While goal not done:\n    think()\n    act()\n    observe()\n    update_memory()\n    ...'

### now define simple agent which understand user query , and plans it , then makes a tool call and return result
- query check math operation requested , extract numbers , then call math function by passing numbers
 then return the result of the tool to the user

In [2]:
# simple math agent
def simple_math_agent(goal):
  print("Goal is: ", goal)
  memory=[]
  step=0
  while True:
    # square is an operation
    if "square" in goal:
      number=int(goal.split("square of")[1].split()[0])
      memory.append(f"found the number{number}")
      result = number*number
      memory.append(f"computed the result{result}")
      print("Agent memory:" , memory)
      print("final answer:" , result)
      break
      # 3 times iterating after giving the final result
    else:
      print("this operation is not found")# if square not present then
      break
    step+=1

In [3]:
# prompt = "find the square of 5"
# simple_math_agent(prompt)

In [4]:
# adding LLM as a agent for understanding the user query and calling the tools
from transformers import AutoTokenizer , AutoModelForCausalLM
import torch
model_name = 'Qwen/Qwen2.5-1.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype='auto' , device_map='auto')


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [5]:
def simple_math_agent_llm(goal):
  print("goal is:" , goal)
  prompt = f"""Extract just the numbers to square from the given text:{goal}" ...
  Reply with ONLY the number , or "none" if there are no numbers"""
  messages = [{"role" : "user" , "content" :prompt}]
  text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
  print(text)
  inputs = tokenizer(text, return_tensors="pt")
  inputs = inputs.to(model.device)
  output = model.generate(**inputs , max_new_tokens=100 , do_sample=True)
  full_llm_result = tokenizer.decode(output[0])
  print("output is: " , full_llm_result)

  # Extract only the assistant's reply (the number) from the full LLM output
  # We look for the last occurrence of '<|im_start|>assistant' and then extract until '<|im_end|>'
  try:
    assistant_start_tag = "<|im_start|>assistant\n"
    assistant_start_index = full_llm_result.rfind(assistant_start_tag)
    if assistant_start_index != -1:
      extracted_content = full_llm_result[assistant_start_index + len(assistant_start_tag):].strip()
      assistant_end_index = extracted_content.find("<|im_end|>")
      if assistant_end_index != -1:
        extracted_number_str = extracted_content[:assistant_end_index].strip()
      else:
        extracted_number_str = extracted_content.strip()
    else:
      extracted_number_str = ""
  except Exception as e:
    print(f"Error parsing LLM output: {e}")
    extracted_number_str = ""

  # compute operation
  if extracted_number_str.isdigit():
    number = int(extracted_number_str)
    result = number * number
    print("square of given number is: " , result)
  else:
    print("operation not found or number not extracted correctly")

In [6]:
prompt = "find the square of 5"
simple_math_agent_llm(prompt)

goal is: find the square of 5
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Extract just the numbers to square from the given text:find the square of 5" ...
  Reply with ONLY the number , or "none" if there are no numbers<|im_end|>
<|im_start|>assistant

output is:  <|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Extract just the numbers to square from the given text:find the square of 5" ...
  Reply with ONLY the number , or "none" if there are no numbers<|im_end|>
<|im_start|>assistant
25<|im_end|>
square of given number is:  625


### A simple travel Agent
 where you can ask about
- city information tool
- get weather info
- budget estimation
- currency conversion
- select tool based on user query
- tool execution
- tool result passed to LLM and then final answer
- test

In [7]:
# adding LLM as a agent for understanding the user query and calling the tools
from transformers import AutoTokenizer , AutoModelForCausalLM
import torch
model_name = 'Qwen/Qwen2.5-1.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype='auto' , device_map='auto')


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

###Define the tools

In [8]:
!pip install -q langchain_core

In [9]:
from langchain_core.tools import tool

In [10]:
#1 weather tool
@tool
def weather(City:str):
  """Returns the current weather for a given city."""
  data={"Bangalore":"30C, cloudy", "Udaipur": "35C, Normal", "Mumbai":"25C Rainy"}
  return data.get(City, "weather data not found")

In [11]:
import langchain_core
# weather data
@langchain_core.tools.tool
def Cityinfo(City:str):
  """Returns the city information for a given city."""
  data= {"Bangalore" : "Bangalore is Tech Hub of India ,lots of hotels and shopping malls are available",
  "Udaipur": "City of lakes , Lake Palace is Popular tourist destination", "Mumbai":"Mumbai is city of dreams for film makers,artists and actors"}
  return data.get(City , "city info data not found")

In [12]:
# currency Exchange tool
@tool
def currency(amount:float , from_currency:str , to_currency:str):
  """Convert the amount from one currency to another."""
  if from_currency == "USD" and to_currency == "INR":
    return amount * 95
  elif from_currency == "EURO" and to_currency == "INR":
    return amount * 105
  elif from_currency == "INR" and to_currency == "USD":
    return amount / 95

In [13]:
# store all the tools in list
tools = [weather , Cityinfo , currency]

In [14]:
print(weather.invoke("Udaipur"))

35C, Normal


In [15]:
# cross check an example
print(weather.invoke("Bangalore"))
print(Cityinfo.invoke("Mumbai"))
print(currency.invoke({"amount":100 , "from_currency":"USD" , "to_currency":"INR"}))
#

30C, cloudy
Mumbai is city of dreams for film makers,artists and actors
9500.0


In [16]:
# tools store to dictionary
tools ={"weather": weather , "Cityinfo": Cityinfo , "currency" : currency}

In [17]:
# now create prompt construction tokenize, extract city and intent from the
def ask_model(prompt):
  instruction = """You are a travel agent"
  Available tools:
  weather(city)
  Cityinfo(city)
  currency(amount , from_currency , to_currency)
  If a tool is required , return ONLY JSON.
  Example :
  [{'tool':"weather" , "args" : {"city" : "Udaipur"}}.
  {"tool":Cityinfo , "args":{"city" :"Udaipur"}}]

  If no tool is needed
  [{"tool":None}]"""
  messages = [{"role" : "system", "content" : instruction} , {"role" : "user" , "content" : prompt}]
  # apply tokenizer
  text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
  inputs = tokenizer(text, return_tensors="pt")
  inputs = inputs.to(model.device)
  outputs= model.generate(**inputs, max_new_tokens=200 , do_sample=True)
  # get results
  result = outputs[0][inputs["input_ids"].shape[1]:]
  return tokenizer.decode(result).strip()

In [18]:
# now we need to add LLM,and retrieve
# to get result in JSON format define a function to search for tool and return results in json
import json
import re
def parse_tool_calls(model_reply):
  match= re.search(r"\[.*\]" , model_reply , re.DOTALL)
  if not match:
    return[]
  try:
    calls = json.loads(match.group(0))
  except json.JSONDecodeError:
    return[]
  return [c for c in calls if c.get("tool")]
  # we are using regex to find match

In [19]:
# actions : runing the tools requested and collecting the results
def execute_tools(tool_calls):
  results =[]
  for call in tool_calls:
    name = call.get("tool")
    args = call.get("args", {})
    if name in tools:
      try:
        result = tools[name].invoke(args)
      except Exception as e:
        result = f"error calling the {name}:{e} "
    else:
      result = f"unknown tool {name}"
    results.append({"tool":name , "args" : args , "result" : result})
  return results

In [31]:
import json
import re
# now create full agent looop using llm and above funtions
def run_travel_agent(prompt):
  print("user prompt:", prompt)
  initial_reply = ask_model(prompt) # Store initial reply from LLM for tool identification
  print("reply= ", initial_reply)
  tool_calls = parse_tool_calls(initial_reply)

  if tool_calls: # Tools were identified and parsed successfully
    tool_results = execute_tools(tool_calls)
    print("tool results=", tool_results)
    final_ans = ask_final_model(tool_results=tool_results, original_prompt=prompt, llm_initial_reply=initial_reply)
  else: # No tools were identified or `[{"tool": null}]` was returned or JSON was invalid
    # In this case, `parse_tool_calls` would return an empty list.
    # We still want `ask_final_model` to produce a user-friendly response, potentially using the LLM's raw initial reply.
    final_ans = ask_final_model(tool_results=[], original_prompt=prompt, llm_initial_reply=initial_reply)
  return final_ans

In [ ]:
# Final answer generation using LLM + tool results
def ask_final_model(tool_results, original_prompt, llm_initial_reply):
  """Generate final answer from tool results and original prompt."""
  if not tool_results:
    # No tools used, ask LLM to answer directly
    instruction = f"You are a helpful travel assistant. User asked: {original_prompt}. Provide a helpful answer."
    messages = [{"role": "system", "content": instruction}]
  else:
    # Build context from tool results
    context_lines = []
    for r in tool_results:
      context_lines.append(f"{r['tool']} with {r['args']} returned: {r['result']}")
    context = "\n".join(context_lines)
    instruction = f"You are a helpful travel assistant. Use the tool results below to answer.\nTool Results:\n{context}\nUser query: {original_prompt}\nProvide a concise friendly final answer."
    messages = [{"role": "system", "content": instruction}]
  text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
  inputs = tokenizer(text, return_tensors="pt").to(model.device)
  outputs = model.generate(**inputs, max_new_tokens=250, do_sample=False, pad_token_id=tokenizer.eos_token_id)
  result = outputs[0][inputs["input_ids"].shape[1]:]
  return tokenizer.decode(result, skip_special_tokens=True).strip()


- send the prompt
- extract the input
- tools parsing and into the json format
- execute tools ,


In [23]:
# run_travel_agent("tell me about Udaipur")

user prompt:  tell me about Udaipur
 reply =  [{"tool":"Cityinfo", "args":{"city": "Udaipur"}}]<|im_end|>
tool results=  [{'tool': 'Cityinfo', 'args': None, 'result': 'error calling the Cityinfo:1 validation error for Cityinfo\n  Input should be a valid dictionary or instance of Cityinfo [type=model_type, input_value=None, input_type=NoneType]\n    For further information visit https://errors.pydantic.dev/2.13/v/model_type '}]


'[{"tool":"Cityinfo","args":{"city":"Udaipur"},"result":"error calling the Cityinfo:1 validation error for Cityinfo\\n  Input should be a valid dictionary or instance of Cityinfo [type=model_type, input_value=None, input_type=NoneType]\\n    For further information visit https://errors.pydantic.dev/2.13/v/model_type "}]}<|im_end|>'

In [32]:
run_travel_agent("tell me about Udaipur and convert 100 USD to INR")

user prompt: tell me about Udaipur and convert 100 USD to INR
reply=  [
  {
    "tool":"Cityinfo",
    "args":{
      "city":"Udaipur"
    }
  },
  {
    "tool":"currency",
    "args":{
      "amount":100,
      "from_currency":"USD",
      "to_currency":"INR"
    }
  }
]<|im_end|>


NameError: name 'parse_tools_calls' is not defined